# LLM-as-a-Judge Video Evaluation Demo

This notebook generates a **synthetic CSV** and then evaluates two model outputs using:

- Traditional metrics (classification macro precision/recall/F1 + accuracy)
- ROUGE-L (reference-based text similarity)
- An **LLM-as-a-judge stub** (replaceable with a real judge model/API)
- Paired **bootstrap confidence intervals** and per-scenario reporting

Useful for a 45-minute interview: define schema → compute metrics → slice by scenarios → quantify uncertainty → do failure analysis.

In [25]:
import pandas as pd
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, cohen_kappa_score
from sklearn.utils import resample
from nltk.translate.bleu_score import sentence_bleu
from rouge_score import rouge_scorer

# 1. LOAD DATA (Simulating a Multimodal/LLM Output CSV)
data = {
    'video_id': ['v1', 'v2', 'v3', 'v4', 'v5'],
    'category': ['High-Motion', 'Low-Light', 'High-Motion', 'Low-Light', 'Indoor'],
    'reference': ["A cat jumps on a sofa.", "A person walks in a dark room.", "A car racing fast.", "A blurry figure moves.", "A chef cooks."],
    'model_out': ["A cat jumps on a couch.", "A person is in a room.", "A car is driving.", "A shadow moves.", "A chef prepares food."],
    'human_label': [1, 1, 1, 0, 1],  # 1 = Accurate, 0 = Inaccurate
    'judge_label': [1, 1, 0, 0, 1]
}
df = pd.DataFrame(data)

In [26]:
df.head()

,video_id,category,reference,model_out,human_label,judge_label
0,v1,High-Motion,A cat jumps on a sofa.,A cat jumps on a couch.,1,1
1,v2,Low-Light,A person walks in a dark room.,A person is in a room.,1,1
2,v3,High-Motion,A car racing fast.,A car is driving.,1,0
3,v4,Low-Light,A blurry figure moves.,A shadow moves.,0,0
4,v5,Indoor,A chef cooks.,A chef prepares food.,1,1


In [30]:
df.drop_duplicates(inplace=True)

In [31]:
df.isnull().sum()

video_id       0
category       0
reference      0
model_out      0
human_label    0
judge_label    0
dtype: int64

In [34]:
df.describe(include = 'all')

,video_id,category,reference,model_out,human_label,judge_label
count,5,5,5,5,5.000000,5.000000
unique,5,3,5,5,NaN,NaN
top,v1,High-Motion,A cat jumps on a sofa.,A cat jumps on a couch.,NaN,NaN
freq,1,2,1,1,NaN,NaN
mean,NaN,NaN,NaN,NaN,0.800000,0.600000
std,NaN,NaN,NaN,NaN,0.447214,0.547723
min,NaN,NaN,NaN,NaN,0.000000,0.000000
25%,NaN,NaN,NaN,NaN,1.000000,0.000000
50%,NaN,NaN,NaN,NaN,1.000000,1.000000
75%,NaN,NaN,NaN,NaN,1.000000,1.000000


In [ ]:
# nlp
    - exact match
    - bleu/rouge
# stats
    - mean
    - CI

# human vs judge
    - kappa
# failure analysis
    - length

In [ ]:
def bleu_score(row):
    return sentence_bleu(row['reference'], row['model_out'])
# BLEU measures how much the generated text overlaps with a reference text using n-gram precision.
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
def rough_score(row):
    return scorer.score(row['reference'], row['model_out'])
# ROUGE measures how much of the reference text is captured by the generated text using n-gram recall.

    

In [54]:
exact_match = df[df['reference'] == df['model_out']]

In [55]:
exact_match

,video_id,category,reference,model_out,human_label,judge_label,bleu,rouge


In [52]:
df['bleu'] = df.apply(bleu_score,axis= 1)
df['rouge'] = df.apply(rough_score,axis = 1)

In [53]:
df

,video_id,category,reference,model_out,human_label,judge_label,bleu,rouge
0,v1,High-Motion,A cat jumps on a sofa.,A cat jumps on a couch.,1,1,1.579655e-231,"{'rougeL': (0.8333333333333334, 0.833333333333..."
1,v2,Low-Light,A person walks in a dark room.,A person is in a room.,1,1,1.565662e-231,"{'rougeL': (0.8333333333333334, 0.714285714285..."
2,v3,High-Motion,A car racing fast.,A car is driving.,1,0,1.595497e-231,"{'rougeL': (0.5, 0.5, 0.5)}"
3,v4,Low-Light,A blurry figure moves.,A shadow moves.,0,0,1.556890e-231,"{'rougeL': (0.6666666666666666, 0.5, 0.5714285..."
4,v5,Indoor,A chef cooks.,A chef prepares food.,1,1,1.474056e-231,"{'rougeL': (0.5, 0.6666666666666666, 0.5714285..."


In [57]:
bleu_mean = df['bleu'].mean()
bleu_mean

np.float64(1.5543521620385862e-231)

In [59]:
rouge_mean = df['rouge'].apply(lambda x: x.values[0]).mean()
rouge_mean

TypeError: 'builtin_function_or_method' object is not subscriptable

In [ ]:
ROUGE # overlap metrics
BertScore # semantic similarity output vs ground truth
CLIPScore # check video and text 
judge prompt?
slicing 
failure

In [ ]:
from bert_score import score

references = ["A man is reading a book."]
predictions = ["A person is sitting and reading."]

P, R, F1 = score(predictions, references, lang="en", verbose=False)

print(F1.mean().item())

In [64]:
import torch
import clip
from PIL import Image

model, preprocess = clip.load("ViT-B/32")

image = preprocess(Image.open("frame.jpg")).unsqueeze(0)
text = clip.tokenize(["A dog jumps into a pool."])

with torch.no_grad():
    image_features = model.encodea_image(image)
    text_features = model.encode_text(text)

    image_features /= image_features.norm(dim=-1, keepdim=True)
    text_features /= text_features.norm(dim=-1, keepdim=True)

    clip_score = (image_features @ text_features.T).item()

print(clip_score)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Users/zhuangdiezhou/.pyenv/versions/3.9.1/lib/python3.9/runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Users/zhuangdiezhou/.pyenv/versions/3.9.1/lib/python3.9/runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "/Users/zhuangdiezhou/.pyenv/versions/3.9.1/lib/python3.9/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/zhuangdiezhou/.pyenv/versions/3.9.1/lib/python3.9/site-packages/traitlets/con

ModuleNotFoundError: No module named 'clip'

In [ ]:
# text quality
# video vs text alignment
# temporal consistency
# llm as a judge
# slicing

In [69]:
from rouge_score import rouge_scorer
from bert_score import score as bert_score

Fontconfig warning: ignoring C.UTF-8: not a valid language tag
/Users/zhuangdiezhou/.pyenv/versions/3.9.1/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [82]:
def text_quality_nlp(row):
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer = True)
    rouge_score = scorer.score(row['model_out'],row['reference'])['rougeL'].fmeasure

    return rouge_score

In [89]:
def text_quality_sem(row):
    precision, recall, f1 = bert_score([row['model_out']],[row['reference']],lang = 'en', verbose = False)
    return f1[0].item()

In [ ]:
df['rouge_score'] = df.apply(text_quality_nlp, axis = 1)

In [91]:
df['bert_score'] = df.apply(text_quality_sem, axis = 1)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You sho

In [92]:
df

,video_id,category,reference,model_out,human_label,judge_label,bleu,rouge,rouge_score,bert_score
0,v1,High-Motion,A cat jumps on a sofa.,A cat jumps on a couch.,1,1,1.579655e-231,"{'rougeL': (0.8333333333333334, 0.833333333333...",0.833333,0.995601
1,v2,Low-Light,A person walks in a dark room.,A person is in a room.,1,1,1.565662e-231,"{'rougeL': (0.8333333333333334, 0.714285714285...",0.769231,0.945425
2,v3,High-Motion,A car racing fast.,A car is driving.,1,0,1.595497e-231,"{'rougeL': (0.5, 0.5, 0.5)}",0.500000,0.920421
3,v4,Low-Light,A blurry figure moves.,A shadow moves.,0,0,1.556890e-231,"{'rougeL': (0.6666666666666666, 0.5, 0.5714285...",0.571429,0.935792
4,v5,Indoor,A chef cooks.,A chef prepares food.,1,1,1.474056e-231,"{'rougeL': (0.5, 0.6666666666666666, 0.5714285...",0.571429,0.951314


In [93]:
bert_score(['A cat jumps on a couch.'],['A cat jumps on a sofa.	'],lang = 'en', verbose = False)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


(tensor([0.9956]), tensor([0.9956]), tensor([0.9956]))

In [96]:
import torch
import clip
from PIL import Image

In [97]:
def video_text_align(row):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model, preprocess = clip.load("ViT-B/32", device = device)

    image = [preprocess(Image.open(p).unsqueeze(0)) for p in image_path]
    image = torch.cat(image).to(device)
    
    text_input = clip.tokenize([text]).to(device)

    with torch.no_grad():
        image_features = model.encode_image(images)
        text_features = model.encode_text(text_input)

    image_features = image_features/image_features.norm()
    text_features /= text_features.norm()

    similarity_score = (image_features @ text_features.T).mean().item()

    return similarity_score

In [101]:
def temporal_consistensy(row):
    diffs = [abs(frame_score[i] - frame_score[i-1]) for i in range(1, len(row['frame_scores'])) ]
    ave_diff = sum(diffs)/len(diffs) if diffs else 0.0
    return 1/(1+ave_diff)

In [102]:
df['temporal_score'] = df.apply(temporal_consistensy, axis = 1)

KeyError: 'frame_scores'

In [100]:
# llm as a judge
client = OpenAI(api_key =)
def llm_as_a_judge(row):
    prompt = f"""
    Given the video description {row['description']}
    Evaludate the generated text:
    {row['model_output']}

    Score them from 1-5 on
    - relevance
    - coherence
    - factuality
    return json like {{}}
    """
    
    response = client.chat.completions.create(model = 'gpt-4', messages = [
        {"role":'system','content':"you are a strict json_only evaluator"},
        {'role': 'user','content':prompt}
    ], temperature = 0)
    

    return json.loads(response.choices[0].message.content)

SyntaxError: expression cannot contain assignment, perhaps you meant "=="? (1412452361.py, line 2)

In [99]:
df['llm_score'] = df.apply(llm_as_a_judge, axis = 1)

NameError: name 'llm_as_a_judge' is not defined

In [ ]:
def aggregate_score(row):
    return 0.3 * row['rough_score'] + 0.3 * row['bert_score'] + 0.2 *['clip_score']  +0.1 *['temporal_consistensy'] + 0.1* ['llm_score']

In [ ]:
df['final_score'] = df.apply(aggregate_score, axis = 1)

In [103]:
df.groupby(['task_type'])['final_score'].mean()

KeyError: 'task_type'

In [ ]:
rouge = rouge_scorer.RoughScore(['roughL'],use_stemmer =True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
clip_model, preprocess = clip.load('modelname',device = device)

In [ ]:
def text_score_rouge(row):
    rouge_score = rouge.score(pre, ref)['rougeL'].fmeasure
    return rouge_score

In [ ]:
def text_score_bert(row):
    precision, recall, f1 = bert_score([ref],[pref],lang ='en', verbose = True)
    return f1[0].item()

In [ ]:
def video_text(row):
    frame_path = row['']
    text = row['']

    image = torch.cat[preprocess(Image.open(p).unsqueeze(0)) for p in framepaths].to(device)
    text_input = clip.tokenize(text).to(device)

    with torch.no_grad():
        image_feature = clip.model.encode_image(image)
        text_feature = clip.model.encode_text(text_input)

    image_feature = image_feature.norm(dim = -1, keepdim = True)
    text_feature = text_feature.norm(dim = -1, keepdim = True)

    similarity_score = (image_feature @ text_feature.T).mean().item()

    return similarity score


In [ ]:
def llm as a judge

In [ ]:
def temporal_cons():
    diff = [abs(frame[i] - frame[i-1]) for i in range(1, len(frame))]

    avg_diff = sum(diff)/len(diff) if diff else 0

    return 1/(1+avg_diff)